# GRAST — WP02 Colab A100 acceptance

Không train model hoặc tải dataset. Notebook kiểm tra CUDA/A100, snapshot config, log,
RNG checkpoint và tái lập trong các tiến trình mới. Chọn runtime **A100** trước khi chạy.
Nếu không có A100, preflight sẽ fail; không coi T4/L4/CPU là đã nghiệm thu A100.

Nếu code chưa push GitHub: chạy `uv build` trên local, rồi dùng cell dưới upload
`dist/grast-0.1.0.tar.gz`. Hoặc đặt đúng source/commit đã review ở `/content/GRAST`.
Không đặt token hoặc mật khẩu trong notebook. Dùng PyTorch có sẵn; NumPy chỉ đổi nếu chưa đáp ứng package requirement.

Notebook này chưa được chạy trên A100 ở thời điểm tạo; chỉ output thực tế mới là evidence.

In [ ]:
import io
import subprocess
import sys
import tarfile
import tempfile
from pathlib import Path

PROJECT_ROOT = Path("/content/GRAST")
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    from google.colab import files

    uploaded = files.upload()  # Chọn đúng source distribution WP02 đã review.
    archives = [name for name in uploaded if name.endswith(".tar.gz")]
    if len(archives) != 1:
        raise ValueError("Upload chính xác một grast-0.1.0.tar.gz từ local dist/.")
    destination = Path(tempfile.mkdtemp(prefix="grast-wp02-", dir="/content"))
    with tarfile.open(fileobj=io.BytesIO(uploaded[archives[0]]), mode="r:gz") as archive:
        archive.extractall(destination, filter="data")
    candidates = list(destination.glob("*/pyproject.toml"))
    if len(candidates) != 1:
        raise ValueError("Source archive phải chứa một project root duy nhất.")
    PROJECT_ROOT = candidates[0].parent

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
    check=True,
)
print("Source:", PROJECT_ROOT)

## Preflight A100

Giữ các phiên bản thực tế của Colab; chúng được ghi vào metadata. Không so hash với local
CPU/RTX để kết luận bitwise equality giữa phần cứng. Cell sẽ dừng nếu torch/CUDA/A100 không phù hợp.

In [ ]:
import json

ENVIRONMENT_REPORT = PROJECT_ROOT / "artifacts" / "environment_colab_a100.json"
preflight = subprocess.run(
    [
        sys.executable,
        "-m",
        "llm_scratch.cli",
        "check-env",
        "--device",
        "cuda",
        "--require-gpu-name",
        "A100",
        "--output",
        str(ENVIRONMENT_REPORT),
    ],
    text=True,
    capture_output=True,
    cwd=PROJECT_ROOT,
)
print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr)
if preflight.returncode:
    try:
        details = json.loads(preflight.stdout)
        reason = "; ".join(details["validation"]["errors"])
    except (ValueError, KeyError):
        reason = "Không chạy được CLI; xem stdout/stderr phía trên."
    raise RuntimeError(f"A100 preflight failed: {reason}")

## Hai lần gọi smoke độc lập

Mỗi lần gọi tự tạo hai worker mới, với Python hash seed/cuBLAS workspace/threads đặt trước
interpreter startup; kiểm tra cả checkpoint RNG continuation. So sánh hai run độc lập cùng seed.

Mặc định output nằm trong `/content`, mất khi hết phiên. Muốn lưu vào Drive, tự mount Drive
rồi đổi `RUN_ROOT`; cell từ chối Drive path nếu chưa mount. Đây là phép thử RNG + ma trận nhỏ,
không đo chất lượng model, throughput hoặc online KV-cache VRAM.

In [ ]:
import json
import os

RUN_ROOT = PROJECT_ROOT / "artifacts" / "runs"
# Sau khi tự mount Drive, có thể đổi thành:
# RUN_ROOT = Path("/content/drive/MyDrive/GRAST/artifacts/runs")
if str(RUN_ROOT).startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive chưa mount; không tạo thư mục lưu trữ giả.")

command = [
    sys.executable,
    "-m",
    "llm_scratch.cli",
    "smoke",
    "--config",
    str(PROJECT_ROOT / "configs/experiments/colab_a100.toml"),
    "--output-root",
    str(RUN_ROOT),
]
RUNS = []
for _ in range(2):
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError("A100 smoke failed. Xem run.json và logs của run vừa tạo.")
    RUNS.append(Path(json.loads(completed.stdout)["run_directory"]))

results = [json.loads((run / "result.json").read_text()) for run in RUNS]
assert results[0]["probe_sha256"] == results[1]["probe_sha256"]
assert all(item["repeat_equal"] and item["checkpoint_resume_equal"] for item in results)
print(
    json.dumps(
        {
            "a100_smoke_passed": True,
            "two_independent_runs_equal": True,
            "runs": [str(run) for run in RUNS],
            "result": results[0],
        },
        indent=2,
    )
)

## Lưu evidence trước khi kết thúc Colab

Cell này đóng gói hai run cùng preflight report và tải về máy. Giữ archive để đính kèm
bằng chứng WP02, không commit checkpoint/output vào Git.

In [ ]:
import zipfile

from google.colab import files

EVIDENCE = Path("/content") / f"grast-wp02-a100-{RUNS[-1].name}.zip"
with zipfile.ZipFile(EVIDENCE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(ENVIRONMENT_REPORT, "environment_colab_a100.json")
    for run in RUNS:
        for path in sorted(run.rglob("*")):
            if path.is_file():
                archive.write(path, f"runs/{run.name}/{path.relative_to(run).as_posix()}")
files.download(str(EVIDENCE))